# 5.1 Constraint satisfaction problems and arc consistency: practice notebook

Some problems are best written as variables that need values under rules. Learn to formulate map colouring, N queens and job scheduling as CSPs, and to shrink domains with node and arc consistency and the AC-3 algorithm.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Checking a map colouring

In [ ]:
# Map colouring of Australia as a CSP. A full assignment is a solution if no two neighbours share a colour.
neighbours = [("WA", "NT"), ("WA", "SA"), ("NT", "SA"), ("NT", "Q"), ("SA", "Q"),
              ("SA", "NSW"), ("SA", "V"), ("Q", "NSW"), ("NSW", "V")]

def broken(colour):
    bad = []
    for a, b in neighbours:
        if colour[a] == colour[b]:
            bad.append(a + "-" + b)
    return bad

good = {"WA": "red", "NT": "green", "SA": "blue", "Q": "red", "NSW": "green", "V": "red", "T": "green"}
poor = {"WA": "red", "NT": "green", "SA": "blue", "Q": "green", "NSW": "red", "V": "red", "T": "red"}
print("good assignment breaks:", broken(good))
print("poor assignment breaks:", broken(poor))
print("number of full assignments:", 3 ** 7)

### AC-3 on the slide network A < B < C

In [ ]:
# AC-3 on the slide network: A < B and B < C, every domain {1, 2, 3}.
domains = {"A": [1, 2, 3], "B": [1, 2, 3], "C": [1, 2, 3]}

def ok(x, vx, y, vy):                    # is the pair allowed by the constraint between x and y?
    if (x, y) == ("A", "B"): return vx < vy
    if (x, y) == ("B", "A"): return vx > vy
    if (x, y) == ("B", "C"): return vx < vy
    if (x, y) == ("C", "B"): return vx > vy

def revise(x, y):
    removed = False
    for vx in domains[x][:]:              # loop over a copy, because we delete from the list
        support = False
        for vy in domains[y]:
            if ok(x, vx, y, vy):
                support = True
        if not support:
            domains[x].remove(vx)
            print("remove", vx, "from", x, "(no partner in", y + ")")
            removed = True
    return removed

queue = [("A", "B"), ("B", "A"), ("B", "C"), ("C", "B")]
while len(queue) > 0:
    x, y = queue.pop(0)
    if revise(x, y):
        for arc in [("A", "B"), ("B", "A"), ("B", "C"), ("C", "B")]:
            if arc[1] == x and arc[0] != y and arc not in queue:
                queue.append(arc)         # the neighbours of x must be checked again
print(domains)

## Your turn

### Task 1: REVISE and AC-3

Write `ac3(domains, less)`. domains is a dict of lists. less is a list of [X, Y] pairs meaning X &lt; Y. Run AC-3 and return the new domains as a dict of sorted lists, or None if a domain becomes empty.

Hint: Make two arcs per constraint, one "less than" and one "greater than". After a removal from X, put back every arc (Z, X).

In [ ]:
def ac3(domains, less):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("ac3({'A': [1, 2, 3], 'B': [1, 2, 3], 'C': [1, 2, 3]}, [['A', 'B'], ['B', 'C']])", {'A': [1], 'B': [2], 'C': [3]}), ("ac3({'A': [1, 2, 3, 4, 5], 'B': [1, 2, 3, 4, 5], 'C': [1, 2, 3, 4, 5], 'D': [1, 2, 3, 4, 5]}, [['A', 'B'], ['B', 'C'], ['C', 'D']])", {'A': [1, 2], 'B': [2, 3], 'C': [3, 4], 'D': [4, 5]}), ("ac3({'A': [1, 2], 'B': [1, 2], 'C': [1, 2]}, [['A', 'B'], ['B', 'C']])", None), ("ac3({'X': [3], 'Y': [1, 5]}, [['X', 'Y']])", {'X': [3], 'Y': [5]})])

### Task 2: Count the colourings of a map

Write `count_colourings(regions, borders, k)` that returns how many ways the regions can be coloured with k colours so that no two bordering regions share a colour. Try every complete assignment.

Hint: Turn each number from 0 to k<sup>n</sup> − 1 into a colouring by writing it in base k.

In [ ]:
def count_colourings(regions, borders, k):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("count_colourings(['WA', 'NT', 'SA', 'Q', 'NSW', 'V', 'T'], [['WA', 'NT'], ['WA', 'SA'], ['NT', 'SA'], ['NT', 'Q'], ['SA', 'Q'], ['SA', 'NSW'], ['SA', 'V'], ['Q', 'NSW'], ['NSW', 'V']], 3)", 18), ("count_colourings(['A', 'B', 'C'], [['A', 'B'], ['B', 'C'], ['A', 'C']], 2)", 0), ("count_colourings(['A', 'B'], [['A', 'B']], 3)", 6)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: REVISE and AC-3

The third test has 3 variables in a chain but only 2 values, so a domain empties.

In [ ]:
def ac3(domains, less):
    dom = {}
    for v in domains:
        dom[v] = list(domains[v])
    arcs = []
    for x, y in less:
        arcs.append([x, y, "lt"])
        arcs.append([y, x, "gt"])
    queue = list(arcs)
    while len(queue) > 0:
        x, y, kind = queue.pop(0)
        removed = False
        for vx in dom[x][:]:
            support = False
            for vy in dom[y]:
                if (kind == "lt" and vx < vy) or (kind == "gt" and vx > vy):
                    support = True
            if not support:
                dom[x].remove(vx)
                removed = True
        if removed:
            if len(dom[x]) == 0:
                return None
            for arc in arcs:
                if arc[1] == x and arc[0] != y and arc not in queue:
                    queue.append(arc)
    for v in dom:
        dom[v].sort()
    return dom

In [ ]:
check([("ac3({'A': [1, 2, 3], 'B': [1, 2, 3], 'C': [1, 2, 3]}, [['A', 'B'], ['B', 'C']])", {'A': [1], 'B': [2], 'C': [3]}), ("ac3({'A': [1, 2, 3, 4, 5], 'B': [1, 2, 3, 4, 5], 'C': [1, 2, 3, 4, 5], 'D': [1, 2, 3, 4, 5]}, [['A', 'B'], ['B', 'C'], ['C', 'D']])", {'A': [1, 2], 'B': [2, 3], 'C': [3, 4], 'D': [4, 5]}), ("ac3({'A': [1, 2], 'B': [1, 2], 'C': [1, 2]}, [['A', 'B'], ['B', 'C']])", None), ("ac3({'X': [3], 'Y': [1, 5]}, [['X', 'Y']])", {'X': [3], 'Y': [5]})])

### Solution to task 2: Count the colourings of a map

This brute force tries all 2,187 colourings of Australia. Chapter 5.2 does the same job far faster with backtracking.

In [ ]:
def count_colourings(regions, borders, k):
    n = len(regions)
    count = 0
    for code in range(k ** n):
        colour = {}
        rest = code
        for r in regions:
            colour[r] = rest % k
            rest = rest // k
        good = True
        for a, b in borders:
            if colour[a] == colour[b]:
                good = False
        if good:
            count = count + 1
    return count

In [ ]:
check([("count_colourings(['WA', 'NT', 'SA', 'Q', 'NSW', 'V', 'T'], [['WA', 'NT'], ['WA', 'SA'], ['NT', 'SA'], ['NT', 'Q'], ['SA', 'Q'], ['SA', 'NSW'], ['SA', 'V'], ['Q', 'NSW'], ['NSW', 'V']], 3)", 18), ("count_colourings(['A', 'B', 'C'], [['A', 'B'], ['B', 'C'], ['A', 'C']], 2)", 0), ("count_colourings(['A', 'B'], [['A', 'B']], 3)", 6)])